<a href="https://colab.research.google.com/github/pedromilken/Disciplinas-Doutorado/blob/main/aula0_1_regressao_logistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PGC305 - Tópicos especiais em LLM e Deep Learning

## Definição dos dados

In [ ]:
import torch; import sklearn

# 1. Carregar dados
iris = sklearn.datasets.load_iris()
X = iris.data        # 4 features: sépalas e pétalas
y = (iris.target == 1).astype(float)  # 1 se Versicolor, 0 caso contrário

# 2. Preparar dados para pytorch
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

## Definição do modelo e treinamento

In [ ]:
# 3. Definir modelo: regressão logística
modelo = torch.nn.Linear(4, 1)  # 4 features → 1 saída (probabilidade de ser Versicolor)

# 4. Definir função de perda e algoritmo de otimização
funcao_perda = torch.nn.BCEWithLogitsLoss()  # combinação de sigmoid + BCE
optimizer = torch.optim.SGD(modelo.parameters(), lr=0.1)

## Execução do treinamento

In [ ]:
# 5. Treino
for epoch in range(1000):
    optimizer.zero_grad() # reseta gradiente senão acumula
    outputs = modelo(X)
    loss = funcao_perda(outputs, y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Época [{epoch+1}/100], Loss: {loss.item():.4f}")

In [ ]:
with torch.no_grad(): # Desativa gradientes para poupar memória
    predicoes = torch.sigmoid(outputs) > 0.5
    acuracia = (predicoes == y).sum() / y.size(0)
    print(f"Acurácia: {acuracia.item() * 100:.2f}%")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Carregar e Preprocessar Dados
iris = load_iris()
X = iris.data
y = (iris.target == 1).astype(float)

# CORREÇÃO AQUI: o parâmetro correto é test_size
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Normalização
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Converter para Tensores
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# 3. Definir o Modelo (Rede Neural Simples)
modelo = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

# 4. Perda e Otimizador
funcao_perda = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(modelo.parameters(), lr=0.01)

# 5. Loop de Treinamento
epochs = 200
for epoch in range(epochs):
    modelo.train()
    optimizer.zero_grad()

    outputs = modelo(X_train)
    loss = funcao_perda(outputs, y_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        # Cálculo de acurácia de treino para monitorar
        with torch.no_grad():
            pred_train = (torch.sigmoid(outputs) > 0.5).float()
            acc_train = (pred_train == y_train).sum() / y_train.size(0)
            print(f"Época [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, Acc Treino: {acc_train.item():.2%}")

# 6. Avaliação Final (Teste)
modelo.eval()
with torch.no_grad():
    pred_logits = modelo(X_test)
    pred_probs = torch.sigmoid(pred_logits)
    pred_classes = (pred_probs > 0.5).float()

    acuracia = (pred_classes == y_test).sum() / y_test.size(0)
    print(f"\n--- Resultado Final ---")
    print(f"Acurácia no Teste: {acuracia.item():.2%}")